In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
pip install -q peft

In [ ]:
import pandas as pd
from datasets import Dataset
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os
import shutil

# Load dataset
file_path = "" # insert location of training_data.jsonl
df = pd.read_json(file_path, lines=True)
dataset = Dataset.from_pandas(df).train_test_split(test_size=0.1, seed=42)

# Modell-ID (Hugging Face Hub)
model_id = "mistralai/Mistral-7B-v0.3"

# quantization config for 4-bit quantization (BitsAndBytes) 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer and model mit quantization config
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.gradient_checkpointing_enable() # activate gradient checkpointing for memory efficiency
model = prepare_model_for_kbit_training(model) # prepare the model for k-bit training

# LoRA configuration
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Get the PEFT model with LoRA applied to the specified target modules
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# tokenization function to format the input as "Question: {prompt}\nAnswer: {answer}" and tokenize it
def tokenize_function(examples):
    texts = [f"Question: {p}\nAnswer: {a}{tokenizer.eos_token}" for p, a in zip(examples["prompt"], examples["answer"])]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Training arguments
args = TrainingArguments(
    output_dir="/kaggle/working/results", # insert output directory
    per_device_train_batch_size=2, # small batch size due to memory constraints with 4-bit quantization
    gradient_accumulation_steps=8, # accumulate gradients over 8 steps to simulate a larger batch size
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=5,
    fp16=True,
    report_to="none"
)

# start training
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Start training ...")
trainer.train()

print("Training completed...")

final_path = "/kaggle/working/trained_model" # insert output directory

# create the directory if it doesn't exist
if not os.path.exists(final_path):
    os.makedirs(final_path)

# Save the fine-tuned model and tokenizer
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

# Zip the saved model directory
shutil.make_archive('/kaggle/working/trained_model', 'zip', final_path) # insert output directory
print("Model saved ...")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 13,631,488 || all params: 7,261,655,040 || trainable%: 0.1877


Map:   0%|          | 0/298 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Start training ...


Step,Training Loss,Validation Loss
5,1.862695,1.735762
10,1.706674,1.599609
15,1.539035,1.541520
20,1.568857,1.497677
25,1.426497,1.458836
30,1.405198,1.413598
35,1.373958,1.374415
40,1.333581,1.364995
45,1.302677,1.358263
50,1.250442,1.354062


Training completed...
Model saved ...


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import csv

# turn off training mode
model.eval()

# load dataset
csv_path = "" # insert location of dataset_clean.csv
df = pd.read_csv(csv_path, sep=",", encoding="utf-8-sig")

results = []

print("Start inference ...")

# inference loop
for i, row in df.iterrows():
    try:
        # Prompt-Format wie im Training
        prompt_text = f"Question: {row['prompt']}\nAnswer:"
        
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.3,          # low temperature for more focused answers
                min_new_tokens=20,          # forces the model to write at least 20 tokens
                do_sample=True,
                repetition_penalty=1.1,    # forces the model to avoid repetition and "getting stuck"
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # answer extraction
        if "Answer:" in full_text:
            ans = full_text.split("Answer:")[-1].strip()
        else:
            # Falls "Answer:" fehlt, nimm alles nach der Frage
            ans = full_text[len(prompt_text):].strip()
        
        # if the answer is empty, set a default message
        if not ans:
            ans = "No answer received."
            
        results.append(ans)
        
        if (i + 1) % 10 == 0:
            print(f"Completed: {i + 1}/{len(df)}")

    except Exception as e:
        print(f"Error {i}: {e}")
        results.append("Error")

# save results to CSV
df['result'] = results
df_result = df[['id', 'result']].copy()
df_result.columns = ['id', 'answer']

output_file = "inference_finetuned_model_results.csv"
df_result.to_csv(
    output_file, 
    index=False,           
    sep=',',               
    encoding='utf-8-sig',
    quoting=csv.QUOTE_ALL
)

print(f"Completed: {output_file}")

Start inference ...
Completed: 10/643
Completed: 20/643
Completed: 30/643
Completed: 40/643
Completed: 50/643
Completed: 60/643
Completed: 70/643
Completed: 80/643
Completed: 90/643
Completed: 100/643
Completed: 110/643
Completed: 120/643
Completed: 130/643
Completed: 140/643
Completed: 150/643
Completed: 160/643
Completed: 170/643
Completed: 180/643
Completed: 190/643
Completed: 200/643
Completed: 210/643
Completed: 220/643
Completed: 230/643
Completed: 240/643
Completed: 250/643
Completed: 260/643
Completed: 270/643
Completed: 280/643
Completed: 290/643
Completed: 300/643
Completed: 310/643
Completed: 320/643
Completed: 330/643
Completed: 340/643
Completed: 350/643
Completed: 360/643
Completed: 370/643
Completed: 380/643
Completed: 390/643
Completed: 400/643
Completed: 410/643
Completed: 420/643
Completed: 430/643
Completed: 440/643
Completed: 450/643
Completed: 460/643
Completed: 470/643
Completed: 480/643
Completed: 490/643
Completed: 500/643
Completed: 510/643
Completed: 520/643
C